# Supplementary figures — Figs. S1–S23

Every supplementary figure in this range that is produced in Python, in manuscript order.
The rating-prediction block (**Figs. S24–S33**) is in `SI_figures2.ipynb`; main-text figures
are in `2_code/main/figures/main_figures.ipynb`.

> **Seven SI figures in this range are made in R, not here.** Their scripts sit in this same
> folder: `acf_exposure.R` (Fig. S8), `plot_pretrend_appendix_lpdid_combined_v3.R` (S9),
> `plot_pretrend_appendix_lpdid_combined_v3_gdp.R` (S10),
> `pretrends_timeline_JAM_comparison.R` (S11) and `default_history.R` (S2). Figs. **S13/S14**
> come from `2_code/supplementary/analysis/LP/alternatives/2_LP_TC+TEMP_interactions.R`, which stays with the
> LP analysis because it runs the regressions as well as drawing. Run them with
> `Rscript <file>` from the repository root.

Sections 1–4 repeat the counterfactual construction from the main-figure notebook (~4 minutes)
so this notebook runs on its own.

## 1. Setup — imports and input paths

In [ ]:
import os
from math import exp
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
import scipy.stats as stats
from scipy.stats import norm
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, ListedColormap, BoundaryNorm
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Rectangle
import seaborn as sns
import cartopy.crs as ccrs
import geopandas as gpd
import random

plt.ion()
import os
# portable repo-root resolution (walk up to the .git / .here marker)
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, '.git')) or os.path.isfile(os.path.join(ROOT, '.here'))) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)

# Shared input paths + loaders (2_code/common/paths.py)
import sys
sys.path.insert(0, os.path.join(ROOT, '2_code', 'common'))
from paths import PATHS, coef_dir, coef_file

# Plotting + counterfactual helpers
from coef_loader        import *           # noqa: F401,F403 - coefficient loading
from counterfactual import *           # noqa: F401,F403 - construction + drawing

# Output directories
OUT_DIR    = './4_figures/main/M3_counterfactuals'
M2_OUT_DIR = './4_figures/main/M2_IRF'
M4_OUT_DIR = './4_figures/main/M4_map'
SI_OUT_DIR = './4_figures/supplementary'
print(f'PLOT_START_YEAR={PLOT_START_YEAR}  GREY_END_YEAR={GREY_END_YEAR}')

# --- the LP sensitivity panels additionally need these -----------------------
import matplotlib.gridspec as gridspec
from lp_plots import *                   # noqa: F401,F403 - IRF plotting

COEF_MAIN      = coef_dir('LP_coefficients')
BOOT_TC        = coef_file('bootstrap',      'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_empirical_TC.csv')
BOOT_TEMP      = coef_file('bootstrap',      'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_empirical_TEMP.csv')
BOOT_WILD_TC   = coef_file('bootstrap_wild', 'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_wild_TC.csv')
BOOT_WILD_TEMP = coef_file('bootstrap_wild', 'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_wild_TEMP.csv')

import subprocess, sys
def run_script(path, *args, cwd=None):
    """Run one of the standalone SI generators and show the tail of its output."""
    r = subprocess.run([sys.executable, path, *args], cwd=cwd, capture_output=True, text=True)
    print((r.stdout or r.stderr)[-700:])
    if r.returncode != 0:
        print(f'   [exit {r.returncode}] {path}')
    return r.returncode

## 2. Helper functions — imported from `2_code/common/{coef_loader,lp_plots,counterfactual}.py`

In [ ]:
# All helpers are imported by the setup cell above; nothing here defines a function.
print('helpers loaded from coef_loader.py + lp_plots.py + counterfactual.py')

## 3. Data — every input loaded once, labelled by the figure it feeds

In [ ]:
# LP coefficient tables. The bootstrap-SE runs are appended to TC_main / TEMP_main for Fig. S5 row 1;
# the avg-windspeed runs stay separate because their IRFs use the 'avg_ws' term.

# --- Fig. 2 (a, c) + Fig. S5 + Fig. S6 + Fig. S7 ---------------------------
TC_main   = load_coefficients(COEF_MAIN, shock_type='TC')
TEMP_main = load_coefficients(COEF_MAIN, shock_type='TEMP')

boot_coef_TC        = load_coefficients(BOOT_TC)
boot_coef_TEMP      = load_coefficients(BOOT_TEMP)
boot_coef_wild_TC   = load_coefficients(BOOT_WILD_TC)
boot_coef_wild_TEMP = load_coefficients(BOOT_WILD_TEMP)

TC_main   = _append(TC_main,   boot_coef_TC,        rename_map={'orig_estimate': 'estimate'})
TEMP_main = _append(TEMP_main, boot_coef_TEMP,      rename_map={'orig_estimate': 'estimate'})
TC_main   = _append(TC_main,   boot_coef_wild_TC,   rename_map={'orig_estimate': 'estimate'})
TEMP_main = _append(TEMP_main, boot_coef_wild_TEMP, rename_map={'orig_estimate': 'estimate'})

TC_main, TEMP_main = add_zero_horizon(TC_main, TEMP_main,
                                      terms=['ts_v2', 'cat1_v2', 'cat3_v2'])

# --- Fig. S20 panel a ------------------------------------------------------
TC_avgws   = load_coefficients(coef_dir('alternativeTC'), shock_type='TC')
TEMP_avgws = load_coefficients(coef_dir('alternativeTC'), shock_type='TEMP')
TC_avgws, TEMP_avgws = add_zero_horizon(TC_avgws, TEMP_avgws, terms=['avg_ws'])

print(f'TC_main   {TC_main.shape}   TEMP_main   {TEMP_main.shape}')
print(f'TC_avgws  {TC_avgws.shape}  TEMP_avgws  {TEMP_avgws.shape}')

**Figs. S20 b-d, S21, S22, S23 — counterfactual inputs**

In [ ]:
# Load climate data
TEMP_df = pd.read_csv(PATHS['temp_data'])
TC_df = pd.read_csv(PATHS['tc_data'])
merged_df = pd.merge(TEMP_df, TC_df, on=['ISO', 'year'])

# Load country master list
country_master = pd.read_csv(PATHS['country_master'])
sovereign_ISOs = country_master[country_master['sovereigncountry'] == 1]['ISO']
sovereign_mapping = country_master[['ISO', 'sovereigncountry']].dropna()
mapping_ISOs = pd.concat([sovereign_ISOs, pd.Series(["TWN"])]).reset_index(drop=True).to_list()  # IMF GDD records Taiwan debt separately

# Load economic data
observed_gdp = pd.read_csv(PATHS['gdp_data']).rename(columns={'value': 'gdp_pcap'})
observed_debt = pd.read_csv(PATHS['debt_data']).rename(columns={'public_debt_pp': 'public_pp_gdd'})
observed = pd.merge(observed_gdp, observed_debt, on=['ISO', 'year'], how='left')

# Load coefficients
coef_debt = load_coefficients(PATHS['coef_debt'])
coef_gdp = load_coefficients(PATHS['coef_gdp'])

# Load geographic data
world = gpd.read_file(PATHS['world_shapefile'])

print(f"Data loaded: {len(merged_df)} climate obs, {len(observed)} economic obs, {len(sovereign_ISOs)} ISOs")

In [ ]:
# TC exposure panel + CMIP6 delta panel (attributable-share filters)
_tc_raw = pd.read_csv(PATHS['tc'])
_delta_src = pd.read_csv(PATHS['temp_data'])

In [ ]:
# Alternative TC counterfactual exposure datasets (raw; ISO-mapped in compute)
alternative_tc_counterfactual_files = {
    'avg_windspeed_max_intensity10pct': './1_input/counterfactual_data/avg_windspeed_max_intensity_minus10pct.csv'
}

alternative_tc_raw = {name: pd.read_csv(path)
                      for name, path in alternative_tc_counterfactual_files.items()}

In [ ]:
# SIDS membership from the shared model prep module
import sys, os
_proj_root = os.path.abspath('.')
if _proj_root not in sys.path:
    sys.path.insert(0, _proj_root)
sys.path.insert(0, os.path.join(_proj_root, '2_code', 'common'))
from prep import SIDS_ISOS as _SIDS_ISOS

In [ ]:
# ISO -> nearest TC basin mapping (SI basin boxplots)
iso_basin = pd.read_csv(PATHS['iso_to_basin'])[['ISO', 'nearest_basin']]

In [ ]:
# ENSO phase per year (Nino 3.4) — colors the ranked SI boxplots
enso = pd.read_csv(PATHS['enso_nino34'])
enso = enso[['year', 'nino34_anom_c_mean', 'enso_phase']]
phase_by_year = enso.set_index('year')['enso_phase'].to_dict()
phase_colors = {'El Nino': '#D62728', 'Neutral': '#BDBDBD', 'La Nina': '#1F77B4'}

In [ ]:
# Single-variable avg_ws LP coefficients (alternative exposure metric)
ALT_TC_DIR = coef_dir("alternativeTC") + os.sep
coef_avgws_debt = pd.read_csv(
    f"{ALT_TC_DIR}full_public_pp_GDD_unbalanced_gl2_TC10_T02_TC.csv"
)
coef_avgws_gdp = pd.read_csv(
    f"{ALT_TC_DIR}full_gdp_pcap_wdi_constant2015_USD_unbalanced_gl2_TC10_T02_TC.csv"
)

In [ ]:
map_key = pd.read_csv(PATHS['tc_isomapping'])[['ADMIN', 'ISO_for_mapping']]
ibtracs_path = PATHS.get('ibtracs_data')
if ibtracs_path and os.path.exists(ibtracs_path):
    ibtracs = pd.read_csv(ibtracs_path)
    ibtracs1 = ibtracs.iloc[1:].reset_index(drop=True)
    ibtracs1['SEASON'] = pd.to_numeric(ibtracs1['SEASON'], errors='coerce')
    filtered_ibtracs = ibtracs1[ibtracs1['SEASON'] >= 1990][['SID','NAME','LAT', 'LON', 'BASIN', 'USA_WIND', 'SEASON']]
else:
    filtered_ibtracs = pd.DataFrame(columns=['SID', 'NAME', 'LAT', 'LON', 'BASIN', 'USA_WIND', 'SEASON'])

## 4. Counterfactual construction

Shared by Figs. S20 b-d, S21, S22 and S23.

In [ ]:
# Debt-ratio heatmap grid
grid_debt = create_heatmap_grid(coef_debt, model_type='linear', shock_type='NATH')

years_list = list(grid_debt.pivot(index="area", columns="years", values="value_ts").columns)
desired_ticks = [0, 5, 10, 15]
tick_positions = [i for i, year in enumerate(years_list) if year in desired_ticks]
tick_labels = [int(year) for year in years_list if year in desired_ticks]

ts_dist = TC_df[(TC_df['year'] >= 1980) & (TC_df['year'] < 2020)]['ts_v2'].values
cat1_dist = TC_df[(TC_df['year'] >= 1980) & (TC_df['year'] < 2020)]['cat1_v2'].values
cat3_dist = TC_df[(TC_df['year'] >= 1980) & (TC_df['year'] < 2020)]['cat3_v2'].values
sigma_value = 2.5
colors = ["#B5A8D5", "#7A73D1", "#4D55CC"]

In [ ]:
# GDP-per-capita heatmap grid (sign flipped so damages plot positive)
grid_gdp = create_heatmap_grid(coef_gdp, model_type='linear', shock_type='NATH', sign=-1)

In [ ]:
# Extract coefficients
coefs_debt = extract_model_coefficients(coef_debt)
coefs_gdp = extract_model_coefficients(coef_gdp)

In [ ]:
# PROCESS coefficients (DEBT RATIO)
results_TC = generate_iso_yearly_vectors_onlyTC(merged_df, coefs_debt['ts'], coefs_debt['cat1'], coefs_debt['cat3'])
results_df_onlyTC = process_results(results_TC)

results_TEMP = generate_iso_yearly_vectors_onlyTEMP_delta(merged_df, coefs_debt['temp'], coefs_debt['temp_interaction'])
results_df_onlyTEMP_delta = process_results(results_TEMP)

# PROCESS coefficients (GDP PER CAPITA)
results_TC_gdp = generate_iso_yearly_vectors_onlyTC(merged_df, coefs_gdp['ts'], coefs_gdp['cat1'], coefs_gdp['cat3'])
results_df_onlyTC_gdp = process_results(results_TC_gdp)

results_TEMP_gdp = generate_iso_yearly_vectors_onlyTEMP_delta(merged_df, coefs_gdp['temp'], coefs_gdp['temp_interaction'])
results_df_onlyTEMP_delta_gdp = process_results(results_TEMP_gdp)

In [ ]:
# CALCULATE COUNTERFACTUALS
select_countries = ["USA", "PHL", 'BRB']
combined_TC = calculate_country_counterfactuals(select_countries, observed_debt, results_df_onlyTC, 'public_pp_gdd')
gdp_combined_cf = calculate_combined_counterfactuals(select_countries, observed, results_df_onlyTC_gdp, results_df_onlyTEMP_delta_gdp, 'gdp_pcap')

In [ ]:
# === Extended time series ===

# === debt-ratio time series ===
results_TC_ext        = generate_iso_yearly_vectors_onlyTC(
                            merged_df, coefs_debt['ts'], coefs_debt['cat1'], coefs_debt['cat3'],
                            start_year=PLOT_START_YEAR)
results_df_onlyTC_ext = process_results(results_TC_ext, start_year=PLOT_START_YEAR)

# === GDP time series ===
results_TC_gdp_ext        = generate_iso_yearly_vectors_onlyTC(
                                merged_df, coefs_gdp['ts'], coefs_gdp['cat1'], coefs_gdp['cat3'],
                                start_year=PLOT_START_YEAR)
results_TEMP_gdp_ext              = generate_iso_yearly_vectors_onlyTEMP_delta(
                                        merged_df, coefs_gdp['temp'], coefs_gdp['temp_interaction'],
                                        start_year=PLOT_START_YEAR)
results_df_onlyTC_gdp_ext = process_results(results_TC_gdp_ext, start_year=PLOT_START_YEAR)
results_df_onlyTEMP_delta_gdp_ext = process_results(results_TEMP_gdp_ext, start_year=PLOT_START_YEAR)

In [ ]:
select_countries = ['USA', 'PHL', 'BRB']

In [ ]:
# --- DEBT RATIO extended counterfactual stack ---
combined_TC_ext       = calculate_country_counterfactuals(
                            select_countries, 
                            observed_debt, results_df_onlyTC_ext,
                            'public_pp_gdd',
                            start_year=PLOT_START_YEAR, end_year=2022)

In [ ]:
# --- GDP extended counterfactual (TC + temp combined) ---
gdp_combined_cf_ext = calculate_combined_counterfactuals(
                          select_countries, observed,
                          results_df_onlyTC_gdp_ext, results_df_onlyTEMP_delta_gdp_ext,
                          'gdp_pcap',
                          start_year=PLOT_START_YEAR)

In [ ]:
# === Share of observed debt / GDP attributable to TCs and temperature (per country, 1980-2019) ===
# debt: TC-only counterfactual. gdp: combined TC + temp (TC-only and temp-only shares also kept).

# 1) Build extended counterfactual frames covering ALL countries.
all_isos_debt = sorted(results_df_onlyTC_ext['ISO'].unique())
# Union of TC and temp ISO sets, so the combined counterfactual covers all GDP-eligible
# countries (non-TC countries get cf_TC = 0, i.e. the temperature channel only).
all_isos_gdp  = sorted(set(results_df_onlyTC_gdp_ext['ISO']) |
                       set(results_df_onlyTEMP_delta_gdp_ext['ISO']))

combined_TC_all = calculate_country_counterfactuals(
    all_isos_debt, observed_debt, results_df_onlyTC_ext,
    'public_pp_gdd', start_year=PLOT_START_YEAR, end_year=2022,
)
gdp_combined_all = calculate_combined_counterfactuals(
    all_isos_gdp, observed,
    results_df_onlyTC_gdp_ext, results_df_onlyTEMP_delta_gdp_ext,
    'gdp_pcap', start_year=PLOT_START_YEAR,
)

# 1b) Extratropical countries (lat extent > 23.5 or < -23.5).
_country_bounds = (
    world[['ADM0_A3', 'geometry']]
    .dropna(subset=['ADM0_A3'])
    .assign(_miny=lambda d: d.geometry.bounds['miny'],
            _maxy=lambda d: d.geometry.bounds['maxy'])
    .groupby('ADM0_A3')
    .agg(miny=('_miny', 'min'), maxy=('_maxy', 'max'))
)
extratropical_isos = set(
    _country_bounds.index[(_country_bounds['miny'] >  23.5) |
                           (_country_bounds['maxy'] < -23.5)]
)
# NOR's outlying territories (e.g. Bouvet Island) defeat the lat-bounds rule, so add it by hand.
extratropical_isos |= {'NOR'}
print(f'extratropical_isos: {len(extratropical_isos)} countries')

# 2) Compute share_attributable; only fill 1980-2019.
mask_debt = (combined_TC_all['year'] >= 1980) & (combined_TC_all['year'] <= 2019)
combined_TC_all['share_attributable'] = np.where(
    mask_debt,
    (combined_TC_all['public_pp_gdd'] - combined_TC_all[f'debt_{PLOT_START_YEAR}'])
        / combined_TC_all['public_pp_gdd'],
    np.nan,
)

# GDP, TC and combined: (obs - cf) / obs, so a negative share = climate damage.
mask_gdp = (gdp_combined_all['year'] >= 1980) & (gdp_combined_all['year'] <= 2019)
for suffix in ['TC', 'combined']:
    cf_col = f'{PLOT_START_YEAR}_cf_{suffix}'
    gdp_combined_all[f'share_attributable_{suffix}'] = np.where(
        mask_gdp,
        (gdp_combined_all['gdp_pcap'] - gdp_combined_all[cf_col])
            / gdp_combined_all['gdp_pcap'],
        np.nan,
    )
# GDP, temp-only: (cf - obs) / cf, as in the SI temperature-share panel.
_cf_temp = f'{PLOT_START_YEAR}_cf_temp'
gdp_combined_all['share_attributable_temp'] = np.where(
    mask_gdp,
    (gdp_combined_all[_cf_temp] - gdp_combined_all['gdp_pcap'])
        / gdp_combined_all[_cf_temp],
    np.nan,
)
gdp_combined_all['share_attributable'] = gdp_combined_all['share_attributable_combined']

# 2b) Shock-event filter: ISOs with >=1 year of TS-strength or stronger exposure
# (ts_v2 / cat1_v2 / cat3_v2 > 0) in 1980-2019.
_tc_raw = pd.read_csv(PATHS['tc'])
_tc_panel = _tc_raw[(_tc_raw['year']>=1980) & (_tc_raw['year']<=2019)].copy()
_tc_panel['_shock'] = ((_tc_panel['ts_v2']>0) |
                        (_tc_panel['cat1_v2']>0) |
                        (_tc_panel['cat3_v2']>0))
valid_shock_isos = set(_tc_panel.groupby('ISO')['_shock'].sum()
                                  .pipe(lambda s: s[s>=1].index))
print(f'valid_shock_isos: {len(valid_shock_isos)} ISOs with >=1 shock year (1980-2019)')

# 2c) Keep ISOs with CMIP6 attribution data (`delta` non-NaN in 1980-2019); ISOs without it
# (ISR, BMU, CYM, PRI, PSE, TCA, XKX) have no temperature counterfactual and are excluded.
_delta_src = pd.read_csv(PATHS['temp_data'])
_delta_panel = _delta_src[(_delta_src['year']>=1980) &
                            (_delta_src['year']<=2019)]
valid_delta_isos = set(_delta_panel.groupby('ISO')['delta']
                                      .apply(lambda s: s.notna().any())
                                      .pipe(lambda s: s[s].index))
print(f'valid_delta_isos: {len(valid_delta_isos)} ISOs with >=1 valid delta year (1980-2019)')

# 3) Per-country aggregates: median + IQR. Filter to sovereigns.
sov_set = set(sovereign_ISOs)

def country_stats(df, value_col='share_attributable', sort_ascending=True,
                  require_shock=True):
    """Per-country median + IQR. Filters: sovereign, valid CMIP6 attribution
    data (non-NaN delta), n>=5 valid years, and (when require_shock=True) ISO
    has >=1 above-TS shock year in 1980-2019."""
    g = (df[df['ISO'].isin(sov_set) & df['ISO'].isin(valid_delta_isos)]
           .dropna(subset=[value_col])
           .groupby('ISO')
           .agg(median=(value_col, 'median'),
                q25=(value_col, lambda s: s.quantile(0.25)),
                q75=(value_col, lambda s: s.quantile(0.75)),
                n=(value_col, 'count'))
           .reset_index())
    g = g[g['n'] >= 5].copy()
    if require_shock:
        g = g[g['ISO'].isin(valid_shock_isos)].copy()
    g['is_extra'] = g['ISO'].isin(extratropical_isos)
    g = g.sort_values(['is_extra', 'median'], ascending=[False, sort_ascending])
    return g.reset_index(drop=True)

# Debt: TC-only, ascending (most-positive tropical at top).
debt_stats     = country_stats(combined_TC_all,                                   sort_ascending=True)
# GDP TC: drop TC=0 ISOs (those go to plot 3).
gdp_stats_TC   = country_stats(gdp_combined_all, value_col='share_attributable_TC',   sort_ascending=False)
# GDP temp: keep ALL ISOs (incl. zero-temp + extratropical) so plot 2/3 can pull from this.
gdp_stats_temp = country_stats(gdp_combined_all, value_col='share_attributable_temp',
                                sort_ascending=False, require_shock=False)

# share_attributable_temp = (cf - obs) / cf (positive = damage); negated for display so damage
# plots negative, as in the TC and combined panels (the xlabel carries the leading minus).
gdp_stats_temp = gdp_stats_temp.copy()
_old_q25 = gdp_stats_temp['q25'].copy()
gdp_stats_temp['median'] = -gdp_stats_temp['median']
gdp_stats_temp['q25']    = -gdp_stats_temp['q75']
gdp_stats_temp['q75']    = -_old_q25

# Combined-channel stats (TC + temp). The temperature channel affects all countries, so
# require_shock=False. Feeds the `gdp_share_*` columns of attributable_shares.csv.
gdp_stats = country_stats(gdp_combined_all,
                          value_col='share_attributable',
                          sort_ascending=False, require_shock=False)

# ---------- Plot 2: GDP TC + temp paired (TC>0 sovereigns only) ----------
# All ISOs in plot 2 have TC>0. Tropical at top, extratropical at bottom (gray TC).
# Sort descending in df so within each block the highest TC value sits at the
# bottom (y=0 end). Place EXTRATROPICAL block first so it occupies the bottom rows.
gdp_TC_main  = (gdp_stats_TC[~gdp_stats_TC['is_extra']]
                 .sort_values('median', ascending=False)
                 .reset_index(drop=True))
gdp_TC_extra = (gdp_stats_TC[ gdp_stats_TC['is_extra']]
                 .sort_values('median', ascending=False)
                 .reset_index(drop=True))

plot2_tc = pd.concat(
    [gdp_TC_extra[['ISO','median','q25','q75']],
     gdp_TC_main [['ISO','median','q25','q75']]],
    ignore_index=True,
)

# Look up temp share for the plotted ISOs.
temp_lookup = gdp_stats_temp.set_index('ISO')[['median','q25','q75']]
plot2_temp = (temp_lookup
                .reindex(plot2_tc['ISO'].values)
                .reset_index()
                .rename(columns={'index':'ISO'}))

is_extra_arr = ([True] * len(gdp_TC_extra)) + ([False] * len(gdp_TC_main))

# ---------- Plot 3: GDP temp share for ALL TC=0 ISOs (extra + non-extra), 2 panels ----------
# Order rows by central-point-estimate (median temp share) ascending so the
# MOST NEGATIVE values land in the LEFT panel. No TC marker shown.
isos_in_plot2 = set(plot2_tc['ISO'])
gdp_temp_other = (gdp_stats_temp[~gdp_stats_temp['ISO'].isin(isos_in_plot2)]
                   .copy())
gdp_temp_other = gdp_temp_other[gdp_temp_other['median'].abs() > 1e-4].copy()
gdp_temp_other = (gdp_temp_other
                   .sort_values('median', ascending=True)
                   .reset_index(drop=True))

n = len(gdp_temp_other)
half = (n + 1) // 2  # left panel takes the larger half if odd; most-negative half
gdp_temp_left  = gdp_temp_other.iloc[:half].reset_index(drop=True)
gdp_temp_right = gdp_temp_other.iloc[half:].reset_index(drop=True)

In [ ]:
# Save median attributable share + 2019 counterfactual values per country
# (read by generate_XGB_predictions.py, M5_basispoint_plot.py and S_tables).
# Headline columns = combined TC+temp; the *_TC / *_temp columns hold each channel alone.

# (a) Median attributable share per country, debt + GDP side.
debt_save = debt_stats[['ISO', 'median', 'q25', 'q75', 'n']].rename(
    columns={
        'median': 'debt_share_median',
        'q25':    'debt_share_q25',
        'q75':    'debt_share_q75',
        'n':      'debt_share_n',
    }
)
def _gdp_save(stats_df, prefix):
    return stats_df[['ISO', 'median', 'q25', 'q75', 'n']].rename(
        columns={
            'median': f'{prefix}_median',
            'q25':    f'{prefix}_q25',
            'q75':    f'{prefix}_q75',
            'n':      f'{prefix}_n',
        }
    )
gdp_save_combined = _gdp_save(gdp_stats,      'gdp_share')        # headline = combined
gdp_save_TC       = _gdp_save(gdp_stats_TC,   'gdp_share_TC')     # TC-only
gdp_save_temp     = _gdp_save(gdp_stats_temp, 'gdp_share_temp')   # temp-only

attributable_shares = (
    debt_save
    .merge(gdp_save_combined, on='ISO', how='outer')
    .merge(gdp_save_TC,       on='ISO', how='outer')
    .merge(gdp_save_temp,     on='ISO', how='outer')
)

# (b) 2019 observed and counterfactual values for debt-to-GDP and GDP per cap.
debt_2019 = (
    combined_TC_all[combined_TC_all['year'] == 2019]
    [['ISO', 'public_pp_gdd', f'debt_{PLOT_START_YEAR}']]
    .rename(columns={
        'public_pp_gdd':           'debt_observed_2019',
        f'debt_{PLOT_START_YEAR}': 'debt_cf_2019',
    })
)
# GDP: TC, temp and combined as separate columns; headline `gdp_pcap_cf_2019` = combined.
gdp_2019 = (
    gdp_combined_all[gdp_combined_all['year'] == 2019]
    [['ISO', 'gdp_pcap',
      f'{PLOT_START_YEAR}_cf_TC',
      f'{PLOT_START_YEAR}_cf_temp',
      f'{PLOT_START_YEAR}_cf_combined']]
    .rename(columns={
        'gdp_pcap':                       'gdp_pcap_observed_2019',
        f'{PLOT_START_YEAR}_cf_TC':       'gdp_pcap_cf_TC_2019',
        f'{PLOT_START_YEAR}_cf_temp':     'gdp_pcap_cf_temp_2019',
        f'{PLOT_START_YEAR}_cf_combined': 'gdp_pcap_cf_combined_2019',
    })
)
gdp_2019['gdp_pcap_cf_2019'] = gdp_2019['gdp_pcap_cf_combined_2019']  # headline alias

attributable_shares = (
    attributable_shares
    .merge(debt_2019, on='ISO', how='left')
    .merge(gdp_2019,  on='ISO', how='left')
)

attributable_shares['is_extratropical'] = (
    attributable_shares['ISO'].isin(extratropical_isos)
)

out_path = './3_output/counterfactual_data/attributable_shares.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
attributable_shares.to_csv(out_path, index=False)
print(
    f'Saved {len(attributable_shares)} sovereigns to {out_path}\n'
    f'  {(~attributable_shares["is_extratropical"]).sum()} tropical, '
    f'{attributable_shares["is_extratropical"].sum()} extratropical\n'
    f'  Columns: {list(attributable_shares.columns)}'
)

attributable_shares.head()

In [ ]:
# Convert yearly TC counterfactual vectors to a long DataFrame for alternative scenarios
analysis_years = list(range(1990, 2020))
vector_length = len(analysis_years)

results_TC_vectors = yearly_vectors_to_long(results_TC)

tc_maxws_complete = TC_df
tc_maxws_complete = tc_maxws_complete[(tc_maxws_complete['year'] >= 1990) & (tc_maxws_complete['year'] <= 2019)].copy()

debt_lookup = {
    iso: grp.set_index('year').reindex(analysis_years)['public_pp_gdd'].to_numpy(dtype=float)
    for iso, grp in observed_debt[
        (observed_debt['year'] >= 1990) & (observed_debt['year'] <= 2019)
    ][['ISO', 'year', 'public_pp_gdd']].groupby('ISO')
}

results_TC_vectors_merged = build_tc_vectors_merged(results_TC, tc_maxws_complete, debt_lookup)

alternative_tc_exposure_data = {}
alternative_results_TC = {}
alternative_results_TC_vectors_merged = {}
alternative_tc_unmatched_countries = {}

for scenario_name in alternative_tc_counterfactual_files:
    alt_tc_df, unmatched_countries = prepare_alternative_tc_exposure(alternative_tc_raw[scenario_name], country_master)
    alt_results_TC = generate_iso_yearly_vectors_onlyTC(alt_tc_df, coefs_debt['ts'], coefs_debt['cat1'], coefs_debt['cat3'])
    alternative_tc_exposure_data[scenario_name] = alt_tc_df
    alternative_results_TC[scenario_name] = alt_results_TC
    alternative_results_TC_vectors_merged[scenario_name] = build_tc_vectors_merged(alt_results_TC, alt_tc_df, debt_lookup)
    alternative_tc_unmatched_countries[scenario_name] = unmatched_countries

results_TC_vectors_merged_avg_windspeed_max_intensity10pct = alternative_results_TC_vectors_merged['avg_windspeed_max_intensity10pct']

pd.DataFrame([
    {
        'scenario': 'baseline_tc_maxws_complete',
        'n_rows': len(results_TC_vectors_merged),
        'n_isos': results_TC_vectors_merged['ISO'].nunique(),
        'n_unmatched_countries': 0
    },
    *[
        {
            'scenario': scenario_name,
            'n_rows': len(alternative_results_TC_vectors_merged[scenario_name]),
            'n_isos': alternative_results_TC_vectors_merged[scenario_name]['ISO'].nunique(),
            'n_unmatched_countries': len(alternative_tc_unmatched_countries[scenario_name])
        }
        for scenario_name in alternative_tc_counterfactual_files
    ]
])

In [ ]:
# Alternative severity definitions
top5_totalexposed_removed, cumulative_impact_top5_totalexposed_removed, top5_totalexposed_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'totalexposed', 5, debt_lookup
)
top10_totalexposed_removed, cumulative_impact_top10_totalexposed_removed, top10_totalexposed_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'totalexposed', 10, debt_lookup
)
top5_avg_ws_removed, cumulative_impact_top5_avg_ws_removed, top5_avg_ws_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'avg_ws', 5, debt_lookup
)
top10_avg_ws_removed, cumulative_impact_top10_avg_ws_removed, top10_avg_ws_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'avg_ws', 10, debt_lookup
)

alternative_counterfactuals = {
    'top5_totalexposed_removed': cumulative_impact_top5_totalexposed_removed,
    'top10_totalexposed_removed': cumulative_impact_top10_totalexposed_removed,
    'top5_avg_ws_removed': cumulative_impact_top5_avg_ws_removed,
    'top10_avg_ws_removed': cumulative_impact_top10_avg_ws_removed
}
alternative_counterfactual_paths = {
    'top5_totalexposed_removed': top5_totalexposed_removed_paths,
    'top10_totalexposed_removed': top10_totalexposed_removed_paths,
    'top5_avg_ws_removed': top5_avg_ws_removed_paths,
    'top10_avg_ws_removed': top10_avg_ws_removed_paths
}

pd.DataFrame({
    'scenario': list(alternative_counterfactuals.keys()),
    'n_countries': [df['ISO'].nunique() for df in alternative_counterfactuals.values()]
})

In [ ]:
# Compare 2019 debt-ratio impacts across alternate TC counterfactual datasets
alt_select_countries = ['USA','PHL','BRB','DOM']

alt_titles = ['United States', 'The Philippines','Barbados','Dominican Republic', 'Dominica', 'Jamaica']


alternative_results_df_onlyTC = {
    scenario_name: process_results(results_dict)
    for scenario_name, results_dict in alternative_results_TC.items()
}

# Scenarios in display order. (label, kind, source)
# kind='alt'   uses alternative_results_df_onlyTC[source]
# kind='paths' uses a *_removed_paths dataframe directly (debt_1990 at year=2019)
# kind='base'  is the all-TC-removed baseline from results_df_onlyTC
alt_plot_scenarios = [
    ('all TC removed',             'base',   None),
    ('10% reduced intensity',      'alt',    'avg_windspeed_max_intensity10pct'),
    ('Top 5 storm years removed',  'paths',  top5_avg_ws_removed_paths),
    ('Top 10 storm years removed', 'paths',  top10_avg_ws_removed_paths),
]


impact_rows = []
for label, kind, source in alt_plot_scenarios:
    if kind == 'base':
        df_ = impact_for_base(alt_select_countries, observed_debt, results_df_onlyTC)
    elif kind == 'alt':
        df_ = impact_for_alt(source, alt_select_countries, observed_debt, alternative_results_df_onlyTC)
    else:
        df_ = impact_for_paths(source, alt_select_countries, observed_debt)
    df_['scenario'] = label
    impact_rows.append(df_)
alternative_impact_2019 = pd.concat(impact_rows, ignore_index=True)

# Row 2: % of all-TC-removed (impact remaining after the modification)
base_by_iso = (alternative_impact_2019[alternative_impact_2019['scenario'] == 'all TC removed']
               .set_index('ISO')['impact_2019'])
alternative_impact_2019['pct_of_all_tc'] = alternative_impact_2019.apply(
    lambda r: r['impact_2019'] / base_by_iso[r['ISO']] * 100 if base_by_iso[r['ISO']] != 0 else np.nan, axis=1
)
# Row 3 (pp): pp attributable to the modified subset = base_impact_pp - scenario_impact_pp.
# For 'all TC removed' the attributable bar is shown at its full base value (i.e., the total TC impact).
alternative_impact_2019['impact_attributable_pp'] = np.where(
    alternative_impact_2019['scenario'] == 'all TC removed',
    alternative_impact_2019['ISO'].map(base_by_iso),
    alternative_impact_2019['ISO'].map(base_by_iso) - alternative_impact_2019['impact_2019'],
)
# Row 4 (%): share of all-TC-removed impact attributable to the modified subset.
# 'all TC removed' is shown at 100% (the full total).
alternative_impact_2019['pct_attributable'] = np.where(
    alternative_impact_2019['scenario'] == 'all TC removed',
    100,
    100 - alternative_impact_2019['pct_of_all_tc'],
)

In [ ]:
# === Regional counterfactual plot ===

label_remap_r = {
    'all TC removed':              'all TCs',
    'Top 10 storm years removed':  'Top 10\nseasons',
    'Top 5 storm years removed':   'Top 5\nseasons',
    '10% reduced intensity':       '10%\nintensity\nreduced',
}
# Column order (M4 panel f): all TCs, 10% intensity reduced, Top 10 seasons, Top 5 seasons.
# `subset_colors_r` is zipped positionally with this order.
bar_subset_order_r = ['all TCs', '10%\nintensity\nreduced', 'Top 10\nseasons', 'Top 5\nseasons']
subset_colors_r = ['#D9D9D9', '#8C6239', '#CC6600', '#F4B86A']

scenarios_r = [s for s in alt_plot_scenarios if s[0] in label_remap_r]

# --- Build per-country impact across the 4 scenarios for ALL sovereigns ---
all_isos_r = sovereign_ISOs.tolist()

rows_r = []
for label, kind, source in scenarios_r:
    if kind == 'base':
        cf = calculate_country_counterfactuals(all_isos_r, observed_debt, results_df_onlyTC, 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    elif kind == 'alt':
        cf = calculate_country_counterfactuals(all_isos_r, observed_debt, alternative_results_df_onlyTC[source], 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    else:
        obs = observed_debt[(observed_debt['year'] == 2019) & (observed_debt['ISO'].isin(all_isos_r))][['ISO', 'public_pp_gdd']]
        df_ = source[source['year'] == 2019][['ISO', 'debt_1990']].merge(obs, on='ISO', how='inner')
    df_['impact_2019'] = df_['public_pp_gdd'] - df_['debt_1990']
    df_['scenario'] = label_remap_r[label]
    rows_r.append(df_)
all_impact_r = pd.concat(rows_r, ignore_index=True)

# Restrict to non-zero baseline ISOs and attach baseline + per-country attributable columns
base_by_iso_r = all_impact_r[all_impact_r['scenario'] == 'all TCs'].set_index('ISO')['impact_2019']
all_impact_r = all_impact_r[all_impact_r['ISO'].isin(base_by_iso_r[base_by_iso_r.abs() > 1e-6].index)].copy()
all_impact_r['base_iso'] = all_impact_r['ISO'].map(base_by_iso_r)

denom_r = all_impact_r['public_pp_gdd'] - all_impact_r['impact_2019']
all_impact_r['rate_iso'] = np.where(denom_r.abs() > 1e-9, all_impact_r['impact_2019'] / denom_r, np.nan)
base_rate_by_iso_r = all_impact_r[all_impact_r['scenario'] == 'all TCs'].set_index('ISO')['rate_iso']
all_impact_r['base_rate_iso'] = all_impact_r['ISO'].map(base_rate_by_iso_r)

all_impact_r['rate_attributable_country'] = np.where(
    all_impact_r['scenario'] == 'all TCs',
    all_impact_r['base_rate_iso'],
    all_impact_r['base_rate_iso'] - all_impact_r['rate_iso'],
)
all_impact_r['pct_attributable_country'] = np.where(
    all_impact_r['scenario'] == 'all TCs',
    100,
    np.where(all_impact_r['base_iso'].abs() > 1e-9,
             (all_impact_r['base_iso'] - all_impact_r['impact_2019']) / all_impact_r['base_iso'] * 100,
             np.nan),
)

# Attach Region grouping + SIDS membership
sids_col = 'Small.Island.Developing.States..SIDS.'
country_groups_r = (country_master[['ISO', 'Region.Name', sids_col]]
                    .drop_duplicates('ISO'))
all_impact_r = all_impact_r.merge(country_groups_r, on='ISO', how='left')
all_impact_r['region'] = all_impact_r['Region.Name'].fillna('Unknown')
all_impact_r['is_sids'] = all_impact_r[sids_col].fillna(0).astype(int)

# Aggregate per region (n_countries used for panel titles)
agg_r = (all_impact_r.groupby(['region', 'scenario'], as_index=False)
         .agg(n_countries=('ISO', 'nunique')))
# Append synthetic blocks: Global (all), SIDS (is_sids==1), non-SIDS (is_sids==0)
global_block = (all_impact_r.assign(region='Global').groupby(['region', 'scenario'], as_index=False)
                .agg(n_countries=('ISO', 'nunique')))
sids_block   = (all_impact_r[all_impact_r['is_sids'] == 1].assign(region='SIDS')
                .groupby(['region', 'scenario'], as_index=False)
                .agg(n_countries=('ISO', 'nunique')))
nonsids_block = (all_impact_r[all_impact_r['is_sids'] == 0].assign(region='non-SIDS')
                 .groupby(['region', 'scenario'], as_index=False)
                 .agg(n_countries=('ISO', 'nunique')))
agg_r = pd.concat([global_block, sids_block, nonsids_block, agg_r], ignore_index=True)

# Fixed display order: Global, SIDS, non-SIDS, Americas, Asia, Oceania.
# (Africa and Europe panels intentionally dropped.)
region_order_r = ['Global', 'SIDS', 'non-SIDS', 'Americas', 'Asia', 'Oceania']
region_order_r = [g for g in region_order_r if g in set(agg_r['region'])]

In [ ]:
# === Regional counterfactual plot — hybrid metric per scenario ===
# Y-axis: per-scenario rate (x100), metric chosen per scenario:
#   all TCs                        -> base_rate_iso (full TC rate)
#   10% reduced intensity          -> rate_iso (rate with the reduced-intensity storms kept in cf)
#   Top 10 / Top 5 seasons removed -> base_rate_iso - rate_iso (rate attributable to those seasons)


label_remap_v = {
    'all TC removed':              'all TCs',
    '10% reduced intensity':       '10%\nintensity\nreduced',
    'Top 10 storm years removed':  'Top 10\nseasons',
    'Top 5 storm years removed':   'Top 5\nseasons',
}
# Display order: No TCs (full removal) → 10% intensity → Top 10 → Top 5.
bar_subset_order_v = ['all TCs', '10%\nintensity\nreduced',
                       'Top 10\nseasons', 'Top 5\nseasons']
subset_colors_v = ['#D9D9D9', '#8C6239', '#CC6600', '#F4B86A']

scenarios_v = [s for s in alt_plot_scenarios if s[0] in label_remap_v]
all_isos_v = sovereign_ISOs.tolist()

rows_v = []
for label, kind, source in scenarios_v:
    if kind == 'base':
        cf = calculate_country_counterfactuals(all_isos_v, observed_debt,
                                                 results_df_onlyTC, 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    elif kind == 'alt':
        cf = calculate_country_counterfactuals(all_isos_v, observed_debt,
                                                 alternative_results_df_onlyTC[source],
                                                 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    else:
        obs = (observed_debt[(observed_debt['year'] == 2019)
                              & (observed_debt['ISO'].isin(all_isos_v))]
               [['ISO', 'public_pp_gdd']])
        df_ = source[source['year'] == 2019][['ISO', 'debt_1990']].merge(obs, on='ISO', how='inner')
    df_['impact_2019'] = df_['public_pp_gdd'] - df_['debt_1990']
    df_['scenario']    = label_remap_v[label]
    rows_v.append(df_)
all_impact_v = pd.concat(rows_v, ignore_index=True)

# Restrict to ISOs whose 'all TCs' baseline is non-zero.
base_v = all_impact_v[all_impact_v['scenario'] == 'all TCs'].set_index('ISO')['impact_2019']
all_impact_v = all_impact_v[all_impact_v['ISO'].isin(base_v[base_v.abs() > 1e-6].index)].copy()

# rate_iso = impact_2019 / cf_debt_2019, with cf_debt_2019 = public_pp_gdd - impact_2019 (= debt_1990).
all_impact_v['cf_debt_2019'] = all_impact_v['public_pp_gdd'] - all_impact_v['impact_2019']
all_impact_v['rate_iso'] = np.where(
    all_impact_v['cf_debt_2019'].abs() > 1e-9,
    all_impact_v['impact_2019'] / all_impact_v['cf_debt_2019'],
    np.nan,
)
# Per-ISO base_rate (rate_iso under the 'all TCs removed' scenario).
_base_rate = (all_impact_v[all_impact_v['scenario'] == 'all TCs']
                .set_index('ISO')['rate_iso'])
all_impact_v['base_rate_iso'] = all_impact_v['ISO'].map(_base_rate)

# Plot metric: hybrid (see top of cell).
_PARTIAL_BASE_MINUS = {'Top 10\nseasons', 'Top 5\nseasons'}
all_impact_v['plot_rate'] = np.where(
    all_impact_v['scenario'] == 'all TCs',
    all_impact_v['base_rate_iso'],
    np.where(
        all_impact_v['scenario'].isin(_PARTIAL_BASE_MINUS),
        all_impact_v['base_rate_iso'] - all_impact_v['rate_iso'],
        all_impact_v['rate_iso'],
    ),
)

# Region + SIDS membership.
_iso_region = (country_master[['ISO', 'Region.Name']].drop_duplicates('ISO')
                if 'Region.Name' in country_master.columns
                else pd.DataFrame({'ISO': all_isos_v, 'Region.Name': 'Unknown'}))
all_impact_v = all_impact_v.merge(_iso_region, on='ISO', how='left')
all_impact_v['region']  = all_impact_v['Region.Name'].fillna('Unknown')
all_impact_v['is_sids'] = all_impact_v['ISO'].isin(set(_SIDS_ISOS)).astype(int)

In [ ]:
# Attach SIDS flag (drop any existing column first to avoid _x/_y suffixes).
_sids_col = 'Small.Island.Developing.States..SIDS.'
sids_lookup = country_master[['ISO', _sids_col]].drop_duplicates('ISO')
all_impact_ext = (all_impact_r.drop(columns=[_sids_col], errors='ignore')
                              .merge(sids_lookup, on='ISO', how='left'))
all_impact_ext['sids_status'] = np.where(
    all_impact_ext[_sids_col] == 1, 'SIDS', 'Non-SIDS'
)

In [ ]:
# === Supplementary boxplots by TC basin and by region (hybrid metric) ===
# Row 1: rate (all TCs: full rate; 10% reduced: rate under that cf; Top 10/5: rate attributable to those seasons).
# Row 2: % of total impact (10% reduced: share retained; Top 10/5: share attributable).
# The region plot adds SIDS / Non-SIDS panels; Europe and other extratropical sovereigns form "Extratropical".


label_remap = {
    'all TC removed':              'all TCs',
    '10% reduced intensity':       '10%\nintensity\nreduced',
    'Top 10 storm years removed':  'Top 10\nseasons',
    'Top 5 storm years removed':   'Top 5\nseasons',
}
bar_subset_order = ['all TCs', '10%\nintensity\nreduced', 'Top 10\nseasons', 'Top 5\nseasons']
subset_colors = ['#D9D9D9', '#8C6239', '#CC6600', '#F4B86A']

# --- Compute impact per (ISO, scenario) for ALL sovereigns ---
selected_scenarios = [s for s in alt_plot_scenarios if s[0] in label_remap]
all_isos = sovereign_ISOs.tolist()
impact_rows_all = []
for label, kind, source in selected_scenarios:
    if kind == 'base':
        cf = calculate_country_counterfactuals(all_isos, observed_debt,
                                                results_df_onlyTC, 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    elif kind == 'alt':
        cf = calculate_country_counterfactuals(all_isos, observed_debt,
                                                alternative_results_df_onlyTC[source],
                                                'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    else:
        obs = (observed_debt[(observed_debt['year'] == 2019)
                              & (observed_debt['ISO'].isin(all_isos))]
               [['ISO', 'public_pp_gdd']])
        df_ = source[source['year'] == 2019][['ISO', 'debt_1990']].merge(obs, on='ISO', how='inner')
    df_['impact_2019'] = df_['public_pp_gdd'] - df_['debt_1990']
    df_['scenario'] = label_remap[label]
    impact_rows_all.append(df_)
all_impact = pd.concat(impact_rows_all, ignore_index=True)

# Restrict to ISOs whose 'all TCs' baseline impact is non-zero.
base_by_iso_all = all_impact[all_impact['scenario'] == 'all TCs'].set_index('ISO')['impact_2019']
nonzero_isos = base_by_iso_all[base_by_iso_all.abs() > 1e-6].index
all_impact = all_impact[all_impact['ISO'].isin(nonzero_isos)].copy()
all_impact['base_iso'] = all_impact['ISO'].map(base_by_iso_all)

# Per-scenario rate_iso = impact_2019 / cf_2019 = impact_2019 / (obs - impact_2019).
denom = all_impact['public_pp_gdd'] - all_impact['impact_2019']
all_impact['rate_iso'] = np.where(denom.abs() > 1e-9,
                                    all_impact['impact_2019'] / denom, np.nan)
base_rate_by_iso = all_impact[all_impact['scenario'] == 'all TCs'].set_index('ISO')['rate_iso']
all_impact['base_rate_iso'] = all_impact['ISO'].map(base_rate_by_iso)

# Hybrid metric for row 1 (rate).
_PARTIAL_BASE_MINUS = {'Top 10\nseasons', 'Top 5\nseasons'}
all_impact['plot_rate'] = np.where(
    all_impact['scenario'] == 'all TCs',
    all_impact['base_rate_iso'],
    np.where(
        all_impact['scenario'].isin(_PARTIAL_BASE_MINUS),
        all_impact['base_rate_iso'] - all_impact['rate_iso'],
        all_impact['rate_iso'],  # 10% intensity reduced
    ),
)

# Hybrid metric for row 2 (% of total impact).
all_impact['plot_pct'] = np.where(
    all_impact['scenario'] == 'all TCs',
    100.0,
    np.where(
        all_impact['scenario'] == '10%\nintensity\nreduced',
        np.where(all_impact['base_iso'].abs() > 1e-9,
                  all_impact['impact_2019'] / all_impact['base_iso'] * 100,
                  np.nan),
        np.where(all_impact['base_iso'].abs() > 1e-9,
                  (all_impact['base_iso'] - all_impact['impact_2019'])
                  / all_impact['base_iso'] * 100, np.nan),
    ),
)

# Attach grouping columns
country_groups = (country_master[['ISO', 'Region.Name',
                                    'Small.Island.Developing.States..SIDS.']]
                   .drop_duplicates('ISO'))
all_impact = (all_impact
              .merge(iso_basin, on='ISO', how='left')
              .merge(country_groups, on='ISO', how='left'))

# Knutson (2020) basin mapping
KNUTSON_BASIN = {
    'NATL': 'North Atlantic', 'ENP':  'Northeast Pacific',
    'WNP':  'Northwest Pacific', 'NIO':  'North Indian Ocean',
    'AUS':  'Southwest Pacific', 'SP':   'Southwest Pacific',
    'SWI':  'South Indian Ocean',
}
MANUAL_BASIN = {
    'BRB': 'North Atlantic', 'DMA': 'North Atlantic', 'ATG': 'North Atlantic',
    'VCT': 'North Atlantic', 'LCA': 'North Atlantic', 'KNA': 'North Atlantic',
    'GRD': 'North Atlantic',
    'CPV': 'North Atlantic', 'NOR': 'North Atlantic', 'FRA': 'North Atlantic',
    'TON': 'Southwest Pacific', 'WSM': 'Southwest Pacific',
    'MUS': 'South Indian Ocean', 'SYC': 'South Indian Ocean', 'COM': 'South Indian Ocean',
    'FSM': 'Northwest Pacific',
    'JPN': 'Excluded',
}
all_impact['basin'] = (
    all_impact['ISO'].map(MANUAL_BASIN)
    .fillna(all_impact['nearest_basin'].map(KNUTSON_BASIN))
    .fillna('No basin')
)
all_impact['region']      = all_impact['Region.Name'].fillna('Unknown')
all_impact['sids_status'] = np.where(all_impact['Small.Island.Developing.States..SIDS.'] == 1,
                                       'SIDS', 'Non-SIDS')

# Extratropical assignment: reuse `extratropical_isos` from above if defined, else the lat-bounds rule.
try:
    _ = extratropical_isos
except NameError:
    _bounds = (world[['ADM0_A3', 'geometry']]
                .dropna(subset=['ADM0_A3'])
                .assign(_miny=lambda d: d.geometry.bounds['miny'],
                        _maxy=lambda d: d.geometry.bounds['maxy'])
                .groupby('ADM0_A3')
                .agg(miny=('_miny','min'), maxy=('_maxy','max')))
    extratropical_isos = set(_bounds.index[(_bounds['miny']>23.5)
                                            | (_bounds['maxy']<-23.5)]) | {'NOR'}

all_impact['is_extratropical'] = all_impact['ISO'].isin(extratropical_isos)
all_impact['region_or_extra'] = np.where(
    all_impact['is_extratropical'],
    'Extratropical',
    all_impact['region'],
)

# --- Verification print: Europe → all extratropical?  Extras from other regions?
eu_isos    = set(all_impact.loc[all_impact['region'] == 'Europe', 'ISO'])
eu_extra   = eu_isos & set(extratropical_isos)
print(f"Europe ISOs (region): {sorted(eu_isos)}  (n={len(eu_isos)})")
print(f"  ... of which extratropical: {len(eu_extra)} / {len(eu_isos)}")
extra_other = sorted(set(
    all_impact.loc[all_impact['is_extratropical'] & (all_impact['region'] != 'Europe'),
                    'ISO']
))
print(f"Extratropical ISOs originally in non-Europe regions ({len(extra_other)}): {extra_other}")

In [ ]:
tc_with_groups = TC_df[(TC_df['year'] >= 1990) & (TC_df['year'] <= 2019) & (TC_df['sovereigncountry'] == 1)].copy()
tc_with_groups = tc_with_groups.merge(
    country_master[['ISO', 'Small.Island.Developing.States..SIDS.']].drop_duplicates('ISO'),
    on='ISO', how='left'
)
tc_with_groups['sids_status'] = np.where(
    tc_with_groups['Small.Island.Developing.States..SIDS.'] == 1, 'SIDS', 'Non-SIDS'
)
# Restrict to country-years with TS-strength or stronger exposure (any bin > 0), the h=5 rate sample.
tc_with_groups['_bin_sum'] = (tc_with_groups['ts_v2'].fillna(0)
                       + tc_with_groups['cat1_v2'].fillna(0)
                       + tc_with_groups['cat3_v2'].fillna(0))
tc_with_groups = tc_with_groups[tc_with_groups['_bin_sum'] > 0].drop(columns=['_bin_sum'])

In [ ]:
te_groups = TC_df[(TC_df['year'] >= 1990) & (TC_df['year'] <= 2019) & (TC_df['sovereigncountry'] == 1)].copy()
te_groups = te_groups.merge(
    country_master[['ISO', 'Small.Island.Developing.States..SIDS.']].drop_duplicates('ISO'),
    on='ISO', how='left'
)
te_groups['sids_status'] = np.where(
    te_groups['Small.Island.Developing.States..SIDS.'] == 1, 'SIDS', 'Non-SIDS'
)
# Restrict to country-years with TS-strength or stronger exposure (any bin > 0), the h=5 rate sample.
te_groups['_bin_sum'] = (te_groups['ts_v2'].fillna(0)
                       + te_groups['cat1_v2'].fillna(0)
                       + te_groups['cat3_v2'].fillna(0))
te_groups = te_groups[te_groups['_bin_sum'] > 0].drop(columns=['_bin_sum'])

In [ ]:
a_h5 = float(coefs_debt['ts'][5])
b_h5 = float(coefs_debt['cat1'][5])
c_h5 = float(coefs_debt['cat3'][5])

rate_groups = TC_df[(TC_df['year'] >= 1990) & (TC_df['year'] <= 2019)
                     & (TC_df['sovereigncountry'] == 1)].copy()
rate_groups = rate_groups.merge(
    country_master[['ISO', 'Small.Island.Developing.States..SIDS.']].drop_duplicates('ISO'),
    on='ISO', how='left'
)
rate_groups['sids_status'] = np.where(
    rate_groups['Small.Island.Developing.States..SIDS.'] == 1, 'SIDS', 'Non-SIDS'
)
rate_groups['total_rate_h5'] = (
    rate_groups['ts_v2'].fillna(0)   * a_h5
  + rate_groups['cat1_v2'].fillna(0) * b_h5
  + rate_groups['cat3_v2'].fillna(0) * c_h5
) * 100  # convert log-rate to percent so y-axis matches the '% debt ratio' label
# Restrict to country-years with TS-strength or stronger exposure (any bin > 0), the h=5 rate sample.
rate_groups['_bin_sum'] = (rate_groups['ts_v2'].fillna(0)
                             + rate_groups['cat1_v2'].fillna(0)
                             + rate_groups['cat3_v2'].fillna(0))
rate_groups = rate_groups[rate_groups['_bin_sum'] > 0].drop(columns=['_bin_sum']).copy()

In [ ]:
select_countries = ['USA','PHL','BRB']

In [ ]:
# === ALTERNATIVE EXPOSURE METRIC: Debt-to-GDP counterfactual (avg_ws vs three-bin) ===
# Compare main-model three-bin LP counterfactual against a single-variable LP using
# avg_ws (annual mean wind speed across storms)

beta_avgws_debt = (coef_avgws_debt
    .query("model == 'linear' and Tshock_type == 'NATH' and term == 'avg_ws'")
    .sort_values("y_h")["estimate"]
    .values)
print(f"avg_ws (debt) coefficients: {len(beta_avgws_debt)} horizons (h=0..{len(beta_avgws_debt)-1})")


# 1980 baseline for the avg_ws comparison (the extended series above start in 1970)
PLOT_START_YEAR_ALT = 1980
# Generate yearly counterfactual rates for the three plotted countries.
results_avgws_debt = process_results(
    generate_iso_yearly_vectors_avgws(
        merged_df[merged_df["ISO"].isin(select_countries)],
        beta_avgws_debt, exposure_col="avg_ws",
        start_year=PLOT_START_YEAR_ALT, end_year=2022),
    start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_avgws_debt = calculate_country_counterfactuals(
    select_countries, observed_debt, results_avgws_debt,
    "public_pp_gdd", start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_main_debt = calculate_country_counterfactuals(
    select_countries, observed_debt, results_df_onlyTC_ext,
    "public_pp_gdd", start_year=PLOT_START_YEAR_ALT, end_year=2022)

In [ ]:
# === ALTERNATIVE EXPOSURE METRIC: GDP per capita counterfactual (avg_ws vs three-bin) ===
# Same structure as the debt comparison above, applied to log GDP per capita.
# Reuses generate_iso_yearly_vectors_avgws defined in the cell above.

beta_avgws_gdp = (coef_avgws_gdp
    .query("model == 'linear' and Tshock_type == 'NATH' and term == 'avg_ws'")
    .sort_values("y_h")["estimate"]
    .values)
print(f"avg_ws (gdp)  coefficients: {len(beta_avgws_gdp)} horizons (h=0..{len(beta_avgws_gdp)-1})")

results_avgws_gdp = process_results(
    generate_iso_yearly_vectors_avgws(
        merged_df[merged_df["ISO"].isin(select_countries)],
        beta_avgws_gdp, exposure_col="avg_ws",
        start_year=PLOT_START_YEAR_ALT, end_year=2022),
    start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_avgws_gdp = calculate_country_counterfactuals(
    select_countries, observed, results_avgws_gdp,
    "gdp_pcap", start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_main_gdp = calculate_country_counterfactuals(
    select_countries, observed, results_df_onlyTC_gdp_ext,
    "gdp_pcap", start_year=PLOT_START_YEAR_ALT, end_year=2022)

In [ ]:
# === Per-shock-year IRFs: debt (avg_ws vs three-bin) ===
# For each plotted country, pick the top-3 shock years (largest avg_ws value
# in 1970-2014) and plot the LP-implied cumulative % change in debt ratio at
# horizons h=0..15 for that single shock under both exposure metrics.

# Pull the three-bin debt coefficients (linear, NATH).
ts_b   = (coef_debt.query("model == 'linear' and Tshock_type == 'NATH' and term == 'ts_v2'  ")
                   .sort_values("y_h")["estimate"].values)
cat1_b = (coef_debt.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat1_v2'")
                   .sort_values("y_h")["estimate"].values)
cat3_b = (coef_debt.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat3_v2'")
                   .sort_values("y_h")["estimate"].values)
H_debt = min(len(ts_b), len(beta_avgws_debt))   # common horizon range

In [ ]:
# === Per-shock-year IRFs: GDP per capita (avg_ws vs three-bin) ===
# Same per-shock-year view as the debt cell above, applied to GDP per capita
# coefficients and the GDP avg_ws coefficients.

ts_b_g   = (coef_gdp.query("model == 'linear' and Tshock_type == 'NATH' and term == 'ts_v2'  ")
                    .sort_values("y_h")["estimate"].values)
cat1_b_g = (coef_gdp.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat1_v2'")
                    .sort_values("y_h")["estimate"].values)
cat3_b_g = (coef_gdp.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat3_v2'")
                    .sort_values("y_h")["estimate"].values)
H_gdp = min(len(ts_b_g), len(beta_avgws_gdp))

In [ ]:
# Calculate counterfactuals for all countries using helper function
all_countries_debt = calculate_country_counterfactuals(mapping_ISOs, observed_debt, results_df_onlyTC, 'public_pp_gdd')

# Prepare map data using helper function
merged_map = prepare_map_data(world, all_countries_debt, year=2019, observed_col='public_pp_gdd', cf_col='debt_1990')
merged_map = merged_map.merge(map_key, left_on='ADMIN', right_on='ADMIN', how='left') # Add ISO column for mapping
merged_map = merged_map.merge(sovereign_mapping, left_on='ISO_A3', right_on='ISO', how='left') # for sovereign country filtering
merged_map.loc[merged_map['ISO_y'] == 'ATA', 'sovereigncountry'] = 1 # no hashmarks for Antarctica

darker = truncate_colormap(plt.get_cmap('Oranges'), 0.2, 0.8)
gradient_colors = [darker(i) for i in np.linspace(0, 1, 6)]
bins = [0, 10, 20, 30, 40, 50, 60]

map_colors = gradient_colors    # separate name so the heatmap `colors` palette is not overwritten
GnBu_darker2 = ListedColormap(map_colors)
norm = BoundaryNorm(bins, GnBu_darker2.N)
debt_cmap = GnBu_darker2
debt_norm = norm

In [ ]:
# Fill 'diff' values in 'merged_map' so that territories get the same color
canon = (
    merged_map.loc[merged_map["ISO_y"] == merged_map["ISO_for_mapping"], ["ISO_for_mapping", "diff"]]
    .dropna(subset=["diff"])
    .drop_duplicates(subset=["ISO_for_mapping"], keep="first")
    .set_index("ISO_for_mapping")["diff"]
)
# Assign: all rows inherit their ISO_for_mapping's canonical diff; fallback to their own diff if missing
merged_map["diff_adjusted"] = merged_map["ISO_for_mapping"].map(canon).fillna(merged_map["diff"])

In [ ]:
# Generate counterfactuals for all sovereign countries using helper functions
debtratio_cf = calculate_combined_counterfactuals(
    sovereign_ISOs.tolist(), observed,
    results_df_onlyTC, results_df_onlyTEMP_delta,
    'public_pp_gdd'
)

gdp_cf = calculate_combined_counterfactuals(
    sovereign_ISOs.tolist(), observed,
    results_df_onlyTC_gdp, results_df_onlyTEMP_delta_gdp,
    'gdp_pcap'
)

In [ ]:
# Combine and export
debtratio_cf = debtratio_cf.rename(columns=lambda x: x + '_debtratio' if x not in ['ISO', 'year'] else x)
gdp_cf = gdp_cf.rename(columns=lambda x: x + '_gdp' if x not in ['ISO', 'year'] else x)

combined_cf = pd.merge(debtratio_cf, gdp_cf, on=['ISO', 'year'], how='left')

print(f"Results exported: {combined_cf.shape}")

## 5. Figures

### Fig. S5 — `debt_IRF.pdf`

TC impacts on debt ratio: six robustness rows, in published order.

**row 1 — standard errors**

In [ ]:
# Varying by standard errors
fig, axes = sensitivity_irfs_SEs(
    df=TC_main,
    outcome='public_pp_GDD',
    legend_title='standard errors',
    specification=[
                   '1980onwards_gl2_TC10_T02',
                   '1980onwards_gl2_TC10_T02_DK',
                   '1980onwards_gl2_TC10_T02_WHITE',
                   '1980onwards_gl2_TC10_T02_cluster_region',
                   '1980onwards_gl2_TC10_T02_boot_wild'
                   ],  
    labels={'1980onwards_gl2_TC10_T02': 'main model \n(cluster by ISO)',
            '1980onwards_gl2_TC10_T02_DK': 'Driscoll-Kraay SEs',
            '1980onwards_gl2_TC10_T02_WHITE': 'White SEs',
            '1980onwards_gl2_TC10_T02_cluster_region': 'cluster by region',
            '1980onwards_gl2_TC10_T02_boot_empirical': 'cluster bootstrap',
            '1980onwards_gl2_TC10_T02_boot_wild': 'wild cluster bootstrap'
            },
    pct_format=True,
    ylabel= 'Δ debt ratio /\n % area exposed',
    yticks=[-0.01, -0.005, 0, 0.005, 0.01],
    ylabel_fontsize=10
)

# Save this row (the rows of debt_IRF are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_IRF'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_IRF', 'row1_standard_errors.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 2 — controls**

In [ ]:
# Varying by specification
fig, axes = sensitivity_irfs(
    df=TC_main,
    outcome='public_pp_GDD',
    legend_title='controls',
    specification=[
                   '1980onwards_gl2_TC10_T02',
                   '1980onwards_gl2_TC10', # no temperature shocks
                   '1980onwards_gl1_TC10_T02', #1  outcome lag
                   '1980onwards_gl2_TC5_T02', # 5 TC lags
                   '1980onwards_gl2_TC10_T02_yearFEonly', # only year FEs
                   '1980onwards_gl2_TC10_T02_year_regionFEonly'
                   ],  
    labels={'1980onwards_gl2_TC10_T02': 'main model \n(10 TC lags, \n2 Temp lags, \n 2 growth lags, \nISO+year FEs)',
            '1980onwards_gl1_TC10_T02': '1 growth lag',
            '1980onwards_gl2_TC5_T02': '5 TC lags',
            '1980onwards_gl2_TC10': 'no temperature \ncontrols',
            '1980onwards_gl2_TC10_T02_yearFEonly': 'year FEs only',
            '1980onwards_gl2_TC10_T02_year_regionFEonly': 'year + region FEs only'
            },
    pct_format=True,
    ylabel= 'Δ debt ratio /\n % area exposed',
    legend_fontsize=10,
    ylabel_fontsize=10,
    title_fontsize=10,
    xlabel=''
)

# Save this row (the rows of debt_IRF are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_IRF'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_IRF', 'row2_controls.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 3 — subset**

In [ ]:
# Varying by subset
fig, axes = sensitivity_irfs(
    df=TC_main,
    outcome='public_pp_GDD',
    subset=['full', 'remove_outliers', 'rated_Agrade', 'rated_Bgrade', 'non-rated', 'sids'],  
    time_trend='linear',
    specification='1980onwards_gl2_TC10_T02', # main model
    labels={'full': 'Full Sample', 
            'rated': 'rated', 
            'non-rated': 'non-rated', 
            'remove_outliers': 'remove debt outliers',
            'rated_Agrade': 'investment grade',
            'rated_Bgrade': 'speculative grade'},
    pct_format=True,
    ylabel= 'Δ debt ratio /\n % area exposed',
    xlabel=''
)

# Save this row (the rows of debt_IRF are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_IRF'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_IRF', 'row3_subset.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 4 — time trend**

In [ ]:
# Varying by Time Trend
fig, axes = sensitivity_irfs(
    df=TC_main,
    outcome='public_pp_GDD',
    specification='1980onwards_gl2_TC10_T02',
    time_trend=['linear', 'no_trend', 'quadratic'],
    labels={
        'linear': 'main model \n(country-specific \nlinear trend)',
        'no_trend': 'no trend',
        'quadratic': 'quadratic trend'
    },
    pct_format=True,
    ylabel= 'Δ debt ratio /\n % area exposed',
    xlabel=''
)

# Save this row (the rows of debt_IRF are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_IRF'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_IRF', 'row4_time_trend.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 5 — sample period**

In [ ]:
# Varying by time period
fig, axes = sensitivity_irfs(
    df=TC_main,
    outcome='public_pp_GDD',
    legend_title='Sample Period',
    specification=[
                   '1980onwards_gl2_TC10_T02',
                   '1980_2022_gl2_TC10_T02',
                   'unbalanced_gl2_TC10_T02',
                   '40yrs_data_gl2_TC10_T02',
                   '30yrs_data_gl2_TC10_T02'
                   ],  
    labels={'1980onwards_gl2_TC10_T02': 'main model \n(1980 - 2019)',
            '1970onwards_gl2_TC10_T02': '1970 - 2019',
            '1990onwards_gl2_TC10_T02': '1990 - 2019',
            '1980_2022_gl2_TC10_T02': '1980 - 2022',
            'unbalanced_gl2_TC10_T02': 'unbalanced sample',
            '40yrs_data_gl2_TC10_T02': 'countries with \n>40yrs data (n=119)',
            '30yrs_data_gl2_TC10_T02': 'countries with \n>30yrs data (n=142)'
            },
    pct_format=True,
    ylabel= 'Δ debt ratio /\n % area exposed',
    xlabel=''
)

# Save this row (the rows of debt_IRF are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_IRF'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_IRF', 'row5_sample_period.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 6 — debt data source**

In [ ]:
# Varying by outcomes
fig, axes = sensitivity_irfs(
    df=TC_main,
    outcome=['public_pp_GDD',
             'cg_debt_pp_GDD',
             'gg_debt_WEO', # WEO updated to 2025
             'panel_ggdy', # WB Kose et al 2022
             'gg_debt_GMD'
             ],
    legend_title='debt data source',
    specification='1980onwards_gl2_TC10_T02',
    labels={
        'public_pp_GDD': 'main model \n(public debt (GDD))',
        'gg_debt_GDD': 'raw',
        'cg_debt_pp_GDD': 'central government only \n(GDD)',
        'total_debt_GDD': 'total debt',
        'gg_debt_WEO': 'general government \n(WEO)',
        'panel_ggdy': 'general government \n(WB; Kose et al 2022)',
        'gg_debt_GMD': 'government debt (GMD)'
            },
    pct_format=True,
    ylabel= 'Δ debt ratio /\n % area exposed',
    ylim=[-0.01, 0.012]
)

# Save this row (the rows of debt_IRF are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_IRF'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_IRF', 'row6_debt_data_source.png'), dpi=300, bbox_inches='tight')
plt.show()


### Fig. S6 — `debt_temp.png`

Temperature impacts on debt ratio: three rows x 5/15/25 C.

**row 1 — time trends**

In [ ]:
# Varying by Time Trend
fig, axes = sensitivity_temp_irfs(
    df=TEMP_main,
    outcome='public_pp_GDD',
    specification='1980onwards_gl2_TC10_T02',
    legend_title='time trends',
    subset='full',
    time_trend=['linear', 'no_trend', 'quadratic'],
    labels={
        'linear': 'linear trend',
        'no_trend': 'no trend',
        'quadratic': 'quadratic trend'
    },
    pct_format=True,
    ylabel= 'Δ debt ratio /\n 1C shock'
)

# Save this row (the rows of debt_temp are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_temp'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_temp', 'row1_time_trends.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 2 — sample period**

In [ ]:
# Varying by sample period
fig, axes = sensitivity_temp_irfs(
    df=TEMP_main,
    outcome='public_pp_GDD',
    legend_title='sample period',
    subset='full',
    time_trend='linear',
    specification=[
        '1980onwards_gl2_TC10_T02',
        'unbalanced_gl2_TC10_T02',
        '40yrs_data_gl2_TC10_T02'
    ],
    labels={
        '1980onwards_gl2_TC10_T02': '1980-2019',
        'unbalanced_gl2_TC10_T02': 'unbalanced sample',
        '40yrs_data_gl2_TC10_T02': 'countries with >40 yrs data'
    },
    pct_format=True,
    ylabel='Δ debt ratio /\n 1C shock'
)

# Save this row (the rows of debt_temp are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_temp'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_temp', 'row2_sample_period.png'), dpi=300, bbox_inches='tight')
plt.show()


**row 3 — subset**

In [ ]:
# Subset
fig, axes = sensitivity_temp_irfs_subset(
    df=TEMP_main,
    outcome='public_pp_GDD',
    specification='1980onwards_gl2_TC10_T02',
    legend_title='subset',
    subset=['full', 'rated_Agrade','rated_Bgrade','non-oecd','non-rated','sids'],
    colors={'full': '#FEC288', 
            'rated_Agrade': 'darkgreen',
            'rated_Bgrade': 'maroon',
            'non_rated': 'navy', 
            'sids': 'darkred'},
    last_panel_only_subsets=['sids','non-rated'],
    labels={
        'full': 'full sample',
        'rated_Agrade': 'investment grade',
        'rated_Bgrade': 'speculative grade',
        'non-rated': 'non-rated',
        'sids': 'sids'
    },
    pct_format=True,
    ylabel='Δ debt ratio /\n 1C shock',
    ylim=[-0.3, 0.3],
    lines_only=True
)

# Save this row (the rows of debt_temp are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'debt_temp'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'debt_temp', 'row3_subset.png'), dpi=300, bbox_inches='tight')
plt.show()


### Fig. S7 — `gdp_temp.png`

**top panel — temperature shock (NATH)**

In [ ]:
fig, ax = plot_temp_irfs(
    df=TEMP_main,
    outcome='gdp_pcap_wdi_constant2015_USD',
    specification='1970onwards_gl2_TC10_T02',
    subset='full',
    model='linear',
    tshock_type='NATH',
    title="Temperature shock (NATH)",
    ylabel='Δ GDP pcap /\n 1C shock'
)

# Save this row (the rows of gdp_temp are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'gdp_temp'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'gdp_temp', 'top_temperature_shock_NATH.png'), dpi=300, bbox_inches='tight')
plt.show()


**bottom row — NATH / BILAL / quadratic**

In [ ]:
# Varying by temperature-shock definition
fig, axes = sensitivity_temp_irfs(
    df=TEMP_main,
    outcome='gdp_pcap_wdi_constant2015_USD',
    legend_title='Tshock type',
    subset='full',
    time_trend='linear',
    specification='1970onwards_gl2_TC10_T02',
    tshock_type=['NATH', 'BILAL', 'Tshock0'],
    labels={
        'Tshock0': 'quadratic',  # legend label used in the published figure
        '1980onwards_gl2_TC10_T02': '1980-2019',
        'unbalanced_gl2_TC10_T02': 'unbalanced sample',
        '40yrs_data_gl2_TC10_T02': 'countries with >40 yrs data'
    },
    pct_format=True,
    ylabel= 'Δ GDP pcap /\n 1C shock',
    colors={'NATH': '#FEC288', 'BILAL': 'green', 'Tshock0': 'blue'},
)

# Save this row (the rows of gdp_temp are stacked into one image).
os.makedirs(os.path.join(SI_OUT_DIR, 'gdp_temp'), exist_ok=True)
fig.savefig(os.path.join(SI_OUT_DIR, 'gdp_temp', 'bottom_Tshock_type.png'), dpi=300, bbox_inches='tight')
plt.show()


### Fig. S20 — `supp-avgws-comparison.png`

**panel a, left — debt ratio**

In [ ]:
# Multiple subsets
fig, ax = plot_irf_avg_ws(
    df=TC_avgws,
    outcome='public_pp_GDD',
    specification='unbalanced_gl2_TC10_T02',
    subset=['full', 'rated', 'non-rated', 'sids'],
    figsize=(6, 2)
)
fig.savefig("./4_figures/supplementary/debt_alt_avgws_irf_subsets.png",
            dpi=300, bbox_inches="tight")

**panel a, right — GDP per capita**

In [ ]:
# Multiple subsets
fig, ax = plot_irf_avg_ws(
    df=TC_avgws,
    outcome='gdp_pcap_wdi_constant2015_USD',
    specification='unbalanced_gl2_TC10_T02',
    subset=['full', 'rated', 'non-rated', 'sids'],
    labels={
        'full': 'Full Sample', 
        'rated': 'rated', 
        'non-rated': 'non-rated', 
        'sids': 'SIDS'
    },
    figsize=(6, 2),
    ylabel='Δ GDP pcap / windspeed (m/s)'
)
fig.savefig("./4_figures/supplementary/gdp_alt_avgws_irf_subsets.png",
            dpi=300, bbox_inches="tight")

**panel b — per-shock-year IRFs, debt**

In [ ]:
fig, axes = plt.subplots(len(select_countries), 3,
                          figsize=(10, 1.5 * len(select_countries)),
                          sharex=True, sharey='row')
country_titles = {"USA": "United States", "PHL": "The Philippines", "BRB": "Barbados"}

for r, iso in enumerate(select_countries):
    iso_df = merged_df[(merged_df["ISO"] == iso) &
                        (merged_df["year"].between(1970, 2014))].copy()
    iso_df["avg_ws"] = iso_df["avg_ws"].fillna(0)
    top_years = iso_df.nlargest(3, "avg_ws")["year"].tolist()

    for c, yr in enumerate(top_years):
        row = iso_df[iso_df["year"] == yr].iloc[0]
        irf_main = (row["ts_v2"]   * ts_b[:H_debt]
                  + row["cat1_v2"] * cat1_b[:H_debt]
                  + row["cat3_v2"] * cat3_b[:H_debt])
        irf_ws   = row["avg_ws"] * beta_avgws_debt[:H_debt]
        # LP outcome is log(value_{t+h}) - log(value_{t-1}); convert log points to true % change.
        irf_main = (np.exp(irf_main) - 1) * 100
        irf_ws   = (np.exp(irf_ws)   - 1) * 100
        # Anchor the IRF at h=-1 (year before the shock) to 0.
        xs       = np.arange(-1, H_debt)
        irf_main = np.concatenate(([0.0], irf_main))
        irf_ws   = np.concatenate(([0.0], irf_ws))

        ax = axes[r, c]
        ax.axhline(0, color="grey", linewidth=0.6, alpha=0.7)
        ax.plot(xs, irf_main, color="#4169E1", marker="o", markersize=3,
                linewidth=1.5, label="three-bin")
        ax.plot(xs, irf_ws,   color="#D62728", marker="s", markersize=3,
                linewidth=1.5, label="avg_ws")
        ax.set_title(f"{country_titles[iso]} - shock year {int(yr)}\n"
                     f"ts={row['ts_v2']:.1f}, cat1={row['cat1_v2']:.1f}, "
                     f"cat3={row['cat3_v2']:.1f}, avg_ws={row['avg_ws']:.1f}",
                     fontsize=8)
        ax.set_xticks([0, 5, 10, 15])
        ax.set_xticklabels([0, 5, 10, 15])
        if r == len(select_countries) - 1:
            ax.set_xlabel("Horizon (years)", fontsize=8)
        if c == 0:
            ax.set_ylabel("Δ debt ratio (%)", fontsize=7)
        if r == 0 and c == 0:
            ax.legend(fontsize=7, loc="upper left", frameon=False)
        ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Debt ratio IRFs",
             fontsize=11, fontweight="bold", x=0.05, y=1.0, ha="left")
plt.tight_layout()
plt.savefig("./4_figures/supplementary/debt_alt_avgws_irf_per_year.png",
            dpi=300, bbox_inches="tight")
plt.show()

**panel c — per-shock-year IRFs, GDP**

In [ ]:
fig, axes = plt.subplots(len(select_countries), 3,
                          figsize=(10, 1.5 * len(select_countries)),
                          sharex=True, sharey='row')
country_titles = {"USA": "United States", "PHL": "The Philippines", "BRB": "Barbados"}

for r, iso in enumerate(select_countries):
    iso_df = merged_df[(merged_df["ISO"] == iso) &
                        (merged_df["year"].between(1970, 2014))].copy()
    iso_df["avg_ws"] = iso_df["avg_ws"].fillna(0)
    top_years = iso_df.nlargest(3, "avg_ws")["year"].tolist()

    for c, yr in enumerate(top_years):
        row = iso_df[iso_df["year"] == yr].iloc[0]
        irf_main = (row["ts_v2"]   * ts_b_g[:H_gdp]
                  + row["cat1_v2"] * cat1_b_g[:H_gdp]
                  + row["cat3_v2"] * cat3_b_g[:H_gdp])
        irf_ws   = row["avg_ws"] * beta_avgws_gdp[:H_gdp]
        # LP outcome is log(value_{t+h}) - log(value_{t-1}); convert log points to true % change.
        irf_main = (np.exp(irf_main) - 1) * 100
        irf_ws   = (np.exp(irf_ws)   - 1) * 100
        # Anchor the IRF at h=-1 (year before the shock) to 0.
        xs       = np.arange(-1, H_gdp)
        irf_main = np.concatenate(([0.0], irf_main))
        irf_ws   = np.concatenate(([0.0], irf_ws))

        ax = axes[r, c]
        ax.axhline(0, color="grey", linewidth=0.6, alpha=0.7)
        ax.plot(xs, irf_main, color="#4169E1", marker="o", markersize=3,
                linewidth=1.5, label="three-bin")
        ax.plot(xs, irf_ws,   color="#D62728", marker="s", markersize=3,
                linewidth=1.5, label="avg_ws")
        ax.set_title(f"{country_titles[iso]} - shock year {int(yr)}\n"
                     f"ts={row['ts_v2']:.1f}, cat1={row['cat1_v2']:.1f}, "
                     f"cat3={row['cat3_v2']:.1f}, avg_ws={row['avg_ws']:.1f}",
                     fontsize=8)
        ax.set_xticks([0, 5, 10, 15])
        ax.set_xticklabels([0, 5, 10, 15])
        if r == len(select_countries) - 1:
            ax.set_xlabel("Horizon (years)", fontsize=9)
        if c == 0:
            ax.set_ylabel("Δ GDP per capita (%)", fontsize=7)
        if r == 0 and c == 0:
            ax.legend(fontsize=7, loc="lower left", frameon=False)
        ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("GDP per capita IRFs",
             fontsize=12, fontweight="bold", x=0.05, y=1.0, ha="left")
plt.tight_layout()
plt.savefig("./4_figures/supplementary/gdp_alt_avgws_irf_per_year.png",
            dpi=300, bbox_inches="tight")
plt.show()

**panel d, top — debt counterfactual**

In [ ]:
# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 2.8))
titles = ["United States", "The Philippines", "Barbados"]
cf_col = f"debt_{PLOT_START_YEAR_ALT}"

for i, iso in enumerate(select_countries):
    ax = axes[i]
    main = cf_main_debt[(cf_main_debt["ISO"] == iso) & (cf_main_debt["year"] <= 2019)].sort_values("year")
    alt  = cf_avgws_debt[(cf_avgws_debt["ISO"] == iso) & (cf_avgws_debt["year"] <= 2019)].sort_values("year")

    ax.plot(main["year"], main["public_pp_gdd"], color="black", linewidth=2, alpha=0.7, label="observed")
    ax.plot(main["year"], main[cf_col],
            color="#4169E1", linewidth=1.5, linestyle="--",
            label=f"{PLOT_START_YEAR_ALT}_cf (three-bin)")
    ax.plot(alt["year"], alt[cf_col],
            color="#D62728", linewidth=1.5, linestyle="--",
            label=f"{PLOT_START_YEAR_ALT}_cf (avg_ws)")

    if i == 0:
        ax.legend(fontsize="small", loc="lower left", frameon=False)
        ax.set_ylabel("Debt Ratio (%)", fontsize=9)
    ax.set_title(titles[i], fontsize=12, fontweight="bold", loc="left")
    ax.set_xticks(range(PLOT_START_YEAR_ALT, 2021, 10))
    ax.set_xticklabels(range(PLOT_START_YEAR_ALT, 2021, 10), fontsize=10)
    ax.set_xlim(PLOT_START_YEAR_ALT, 2021)
    ax.set_ylim(0, None)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("black")
    ax.spines[["left", "bottom"]].set_linewidth(0.5)
    ax.grid(False)

plt.tight_layout()
plt.savefig("./4_figures/supplementary/debt_alt_avgws_vs_3bin.png",
            dpi=300, bbox_inches="tight")
plt.show()

**panel d, bottom — GDP counterfactual**

In [ ]:
# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 2.8))
titles = ["United States", "The Philippines", "Barbados"]
cf_col = f"debt_{PLOT_START_YEAR_ALT}"  # calculate_country_counterfactuals reuses the debt_ prefix

for i, iso in enumerate(select_countries):
    ax = axes[i]
    main = cf_main_gdp[(cf_main_gdp["ISO"] == iso) & (cf_main_gdp["year"] <= 2019)].sort_values("year")
    alt  = cf_avgws_gdp[(cf_avgws_gdp["ISO"] == iso) & (cf_avgws_gdp["year"] <= 2019)].sort_values("year")

    ax.plot(main["year"], main["gdp_pcap"], color="black", linewidth=2, alpha=0.7, label="observed")
    ax.plot(main["year"], main[cf_col],
            color="#4169E1", linewidth=1.5, linestyle="--",
            label=f"{PLOT_START_YEAR_ALT}_cf (three-bin)")
    ax.plot(alt["year"], alt[cf_col],
            color="#D62728", linewidth=1.5, linestyle="--",
            label=f"{PLOT_START_YEAR_ALT}_cf (avg_ws)")

    if i == 0:
        ax.legend(fontsize="small", loc="lower right", frameon=False)
        ax.set_ylabel("GDP per capita ($2015)", fontsize=9)
    ax.set_title(titles[i], fontsize=12, fontweight="bold", loc="left")
    ax.set_xticks(range(PLOT_START_YEAR_ALT, 2021, 10))
    ax.set_xticklabels(range(PLOT_START_YEAR_ALT, 2021, 10), fontsize=10)
    ax.set_xlim(PLOT_START_YEAR_ALT, 2021)
    ax.set_ylim(0, None)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: "0" if x == 0 else f"{int(x/1000)}k"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("black")
    ax.spines[["left", "bottom"]].set_linewidth(0.5)
    ax.grid(False)

plt.tight_layout()
plt.savefig("./4_figures/supplementary/gdp_alt_avgws_vs_3bin.png",
            dpi=300, bbox_inches="tight")
plt.show()

### Fig. S21 — `supp-share_attributed-TC.png`

In [ ]:
from matplotlib.lines import Line2D
n_rows_max = max(len(debt_stats), len(plot2_tc))
fig_h = max(0.13 * n_rows_max, 3)
fig, (ax_debt, ax_gdp) = plt.subplots(
    1, 2, figsize=(10, fig_h),
    gridspec_kw={'width_ratios': [1, 1.2]},
)

# Panel a: debt TC-attributable share.
_draw_panel(ax_debt, debt_stats, '',
            '(debtratio_observed - counterfactual) / debtratio_observed',
            line_c=TC_LIGHT, mk_c=TC_DARK,
                extratropical_isos=extratropical_isos)
ax_debt.set_title('Share of observed debt ratio attributable to TCs',
                  fontsize=10, fontweight='bold')

# Panel b: GDP TC + temp paired.
_draw_paired_panel(ax_gdp, plot2_tc, plot2_temp, is_extra_arr,
                   xlabel='-(counterfactual - gdp_pcap_observed) / counterfactual')
legend_handles = [
    Line2D([0],[0], color=TC_DARK,     marker='o', linestyle='-', label='TC-attributable'),
    Line2D([0],[0], color=SALMON_DARK, marker='o', linestyle='-', label='Temp-attributable'),
]
ax_gdp.legend(handles=legend_handles, loc='upper right', fontsize=7, frameon=False)
ax_gdp.set_title('Share of observed GDP attributable to TCs and temperature',
                 fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{SI_OUT_DIR}/supp-share_attributed-TC.png',
            dpi=300, bbox_inches='tight')
plt.show()

### Fig. S22 — `supp-share_attributed-gdp-temp.png`

In [ ]:
# ---------- Plot 3 (cont.): GDP temp share panels ----------
fig, axes = plt.subplots(
    1, 2,
    figsize=(9, max(0.13 * max(len(gdp_temp_left), len(gdp_temp_right)), 3)),
    sharex=True,
)
_draw_panel(axes[0], gdp_temp_left, '',
            '-(counterfactual - gdp_pcap_observed) / counterfactual',
            line_c=SALMON_LIGHT, mk_c=SALMON_DARK,
            differentiate_extra=False,
                extratropical_isos=extratropical_isos)
_draw_panel(axes[1], gdp_temp_right, '',
            '-(counterfactual - gdp_pcap_observed) / counterfactual',
            line_c=SALMON_LIGHT, mk_c=SALMON_DARK,
            differentiate_extra=False,
                extratropical_isos=extratropical_isos)
# Invert y so each panel reads ascending TOP-TO-BOTTOM (lowest at top).
for _ax in axes:
    _ax.invert_yaxis()
axes[0].set_title('Share of observed GDP attributable to temperature', fontsize=10, fontweight='bold', loc='left')
plt.tight_layout()
plt.savefig(f'{SI_OUT_DIR}/supp-share_attributed-gdp-temp.png',
            dpi=300, bbox_inches='tight')
plt.show()


print(f'Debt: {len(debt_stats)} sovereigns; '
      f'median range [{debt_stats["median"].min():.3f}, {debt_stats["median"].max():.3f}]')
print(f'GDP plot 2 (TC>0): {len(gdp_TC_main)} tropical + {len(gdp_TC_extra)} extratropical')
print(f'GDP plot 3 (TC=0): {n} sovereigns '
      f'({len(gdp_temp_left)} left / {len(gdp_temp_right)} right)')

### Fig. S23 — `supp-counterfactual-regions.png` + `supp-counterfactual-basins.png`

In [ ]:
# --- Plot 1: by TC basin (Knutson 2020 — six basins) ---
basin_order = ['Global', 'North Atlantic', 'Northwest Pacific', 'Northeast Pacific',
               'North Indian Ocean', 'South Indian Ocean', 'Southwest Pacific',
               'No basin']
basin_order = [b for b in basin_order
               if b == 'Global' or b in all_impact['basin'].unique()]
basin_panels = [('Global', all_impact)]
for b in basin_order[1:]:
    basin_panels.append((b, all_impact[all_impact['basin'] == b]))
_make_two_row_plot(basin_panels, 'Aggregated by TC basin (Knutson 2020)',
                    bar_subset_order, subset_colors, save_path='./4_figures/supplementary/supp-counterfactual-basins.png')


# --- Plot 2: by Region (with SIDS / Non-SIDS panels; Europe → Extratropical;
# other extratropical sovereigns pooled into the Extratropical panel) ---
region_order_pref = ['Global', 'SIDS', 'Non-SIDS',
                      'Africa', 'Americas', 'Asia', 'Oceania', 'Extratropical']
region_avail = set(all_impact['region_or_extra'].unique())
region_order = [r for r in region_order_pref
                 if r in ('Global','SIDS','Non-SIDS') or r in region_avail]

region_panels = []
for panel in region_order:
    if panel == 'Global':
        sub = all_impact
    elif panel == 'SIDS':
        sub = all_impact[all_impact['sids_status'] == 'SIDS']
    elif panel == 'Non-SIDS':
        sub = all_impact[all_impact['sids_status'] == 'Non-SIDS']
    else:
        sub = all_impact[all_impact['region_or_extra'] == panel]
    region_panels.append((panel, sub))
_make_two_row_plot(region_panels, 'Aggregated by Region',
                    bar_subset_order, subset_colors, save_path='./4_figures/supplementary/supp-counterfactual-regions.png')

## 6. Figures from the standalone generators

Each cell runs one script from the repository root; the scripts own their own inputs and output paths.

### Figs. S3 and S4 — not built here

The literature-comparison forest plots are produced outside this package, from
`climate_debt_shocks_replication_S3_S4/`. They are held separately because their
inputs include rendered pages of copyrighted source papers; see that folder's
README.


### Fig. S12 — `contamination_illustration.png`

In [ ]:
run_script('2_code/supplementary/figures/contamination_illustration.py')

### Fig. S15 — `GDP_map.png`

In [ ]:
run_script('2_code/supplementary/figures/GDP_map.py')

### Fig. S16 — `ws_histogram.png`

In [ ]:
run_script('2_code/supplementary/figures/ws_histogram.py')

### Fig. S17 — `temp_histogram.png`

In [ ]:
run_script('2_code/supplementary/figures/temp_histogram.py')

### Fig. S18 — `R_otherdebtmetrics.png`

In [ ]:
run_script('2_code/supplementary/figures/R_otherdebtmetrics.py')

### Fig. S19 — `LP_creditrating.png`

In [ ]:
run_script('2_code/supplementary/figures/LP_creditrating.py')

### Credit-rating ordinal scale (rating_ordinalscale)

In [ ]:
run_script('2_code/supplementary/figures/rating_ordinalscale_table.py')

### Fig. S1 — `financeflows.png`

`sankey_new.ipynb` (this folder) builds the Plotly Sankey and exports it to file. The published figure adds its column headers, box labels and dollar values by hand on top of this render.

In [ ]:
import subprocess, sys, tempfile
# execute into a temp copy so the committed notebook is not rewritten with outputs
_r = subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook',
                     '--execute', '--output-dir', tempfile.gettempdir(), 'sankey_new.ipynb'],
                    cwd='2_code/supplementary/figures', capture_output=True, text=True)
print((_r.stdout or _r.stderr)[-500:])